F2_03_procesamiento_validacio# F2_03 · Procesamiento y validación

## Finalidad

Implementar las reglas acordadas por el equipo de acuerdo a las evidencias del EDA de Preprocesamiento F2_02, transformando los datos crudos de F2_01 en tablas consistentes, trazables y listas para la integración (F2_03 → matriz analítica).

**Criterio rector:** separar siempre el dato observado de la decisión analítica. Los
archivos originales en `data/interim/` no se modifican; las tablas procesadas se generan
en `data/processed/`.

## Entregables de este notebook

- `diputados_procesados`
- `militancias_observadas` (historial normalizado, sin pérdida de evidencia)
- `militancias_analiticas` (línea de tiempo resuelta, sin solapamientos)
- `detalle_votaciones_procesado`
- `proyecto_ley_procesado`
- `reporte_calidad`

## 1. Configuración de imports y detección de rutas

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [2]:
# Carga de datos de votaciones y diputados requiere que el notebook se ejecute desde la carpeta del proyecto.

cwd = Path.cwd().resolve()
candidatos = [cwd] + list(cwd.parents)

BASE_DIR = None

for candidato in candidatos:
    if candidato.name == "F2" and (candidato / "data").exists():
        BASE_DIR = candidato
        break

if BASE_DIR is None:
    for candidato in candidatos:
        posible = candidato / "F2"
        if (posible / "data").exists():
            BASE_DIR = posible
            break

if BASE_DIR is None:
    raise FileNotFoundError(
        "No se pudo localizar la carpeta F2/data. "
        "Ejecuta el notebook desde el repositorio del proyecto."
    )

DATA_DIR = BASE_DIR / "data"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"BASE_DIR: {BASE_DIR}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"PROCESSED_DIR: {PROCESSED_DIR}")

BASE_DIR: C:\Users\joign\OneDrive\Desktop\Magister en Ciencia de Datos e IA\1. Programación para la ciencia de datos\grupo_1_programacion_ciencia_de_datos\F2
DATA_DIR: C:\Users\joign\OneDrive\Desktop\Magister en Ciencia de Datos e IA\1. Programación para la ciencia de datos\grupo_1_programacion_ciencia_de_datos\F2\data
PROCESSED_DIR: C:\Users\joign\OneDrive\Desktop\Magister en Ciencia de Datos e IA\1. Programación para la ciencia de datos\grupo_1_programacion_ciencia_de_datos\F2\data\processed


## 2. Carga de datasets

Se cargan explícitamente las 5 tablas que produjo F2_01, ya exploradas en F2_02. A
diferencia de F2_02 (que descubre archivos automáticamente para inventariarlos), aquí cada
tabla tiene un rol conocido dentro de las reglas de procesamiento que siguen.

In [3]:
df_proyecto = pd.read_csv(INTERIM_DIR / "VotacionesPorProyectoDeLey" / "proyecto_ley.csv")
df_diputados = pd.read_csv(INTERIM_DIR / "diputados.csv")
df_militancias = pd.read_csv(INTERIM_DIR / "militancias.csv")
df_periodos = pd.read_csv(INTERIM_DIR / "periodos.csv")
df_detalle = pd.read_csv(INTERIM_DIR / "votaciones" / "detalle_votaciones.csv")

tablas = {
    "proyecto_ley": df_proyecto,
    "diputados": df_diputados,
    "militancias": df_militancias,
    "periodos": df_periodos,
    "detalle_votaciones": df_detalle,
}

for nombre, df in tablas.items():
    print(f"{nombre}: {df.shape[0]} filas x {df.shape[1]} columnas")

proyecto_ley: 15 filas x 15 columnas
diputados: 157 filas x 13 columnas
militancias: 407 filas x 6 columnas
periodos: 11 filas x 4 columnas
detalle_votaciones: 1993 filas x 20 columnas


## 3. Reporte de calidad (inicialización)

Antes de aplicar cualquier regla, se inicializa la estructura donde se registrarán todas
las observaciones de calidad, ambigüedades y decisiones aplicadas a lo largo del
procesamiento.

In [4]:
reporte_calidad = []


def registrar_observacion(tabla, diputado_id, tipo, descripcion, regla_aplicada=""):
    """Agrega una fila al reporte de calidad, documentando una anomalía y su tratamiento."""
    reporte_calidad.append({
        "tabla": tabla,
        "diputado_id": diputado_id,
        "tipo": tipo,
        "descripcion": descripcion,
        "regla_aplicada": regla_aplicada,
    })

## 4. Normalización general

Aplicación de reglas acordadas por el equipo sobre las 5 tablas:

| # | Decisión | Aplicación |
|---|---|---|
| 1 | Identificadores | Convertir a texto; eliminar espacios; rechazar vacíos |
| 2 | Fechas | Convertir a datetime; no convertibles detienen el proceso |
| 3 | Totales | Convertir a enteros; no admitir valores no numéricos |
| 4 | Nulos válidos | Conservar los nulos permitidos por la fuente; no imputar |

In [5]:
def normalizar_identificador(serie: pd.Series, nombre_columna: str) -> pd.Series:
    """
    Regla #1: convierte a texto, elimina espacios laterales y rechaza vacíos.
    Si tras la limpieza queda algún valor vacío, detiene el procesamiento
    (los identificadores vacíos son un error de origen, no un nulo válido).
    """
    serie_txt = serie.astype(str).str.strip()

    vacios = serie_txt.eq("") | serie_txt.eq("nan")
    if vacios.any():
        raise ValueError(
            f"'{nombre_columna}' tiene {vacios.sum()} valores vacíos tras la normalización. "
            "Los identificadores vacíos no están permitidos (Regla #1)."
        )

    return serie_txt


def normalizar_fecha(serie: pd.Series, nombre_columna: str, permitir_nulos: bool = True) -> pd.Series:
    """
    Regla #2: convierte a datetime. Si aparece algún valor no convertible
    (que no sea un nulo ya presente en origen), detiene el procesamiento.
    """
    fechas = pd.to_datetime(serie, errors="coerce")

    nulos_originales = serie.isna()
    no_convertibles = fechas.isna() & ~nulos_originales

    if no_convertibles.any():
        raise ValueError(
            f"'{nombre_columna}' tiene {no_convertibles.sum()} valores no convertibles a fecha."
        )

    return fechas


def normalizar_total(serie: pd.Series, nombre_columna: str) -> pd.Series:
    """Regla #3: convierte a entero; no admite valores no numéricos."""
    numerica = pd.to_numeric(serie, errors="coerce")

    no_numericos = numerica.isna() & serie.notna()
    if no_numericos.any():
        raise ValueError(
            f"'{nombre_columna}' tiene {no_numericos.sum()} valores no numéricos."
        )

    return numerica.astype("Int64")

### 4.1 · Diputados

In [6]:
df_diputados_norm = df_diputados.copy()

df_diputados_norm["diputado_id"] = normalizar_identificador(df_diputados_norm["diputado_id"], "diputado_id")
df_diputados_norm["periodo_id"] = normalizar_identificador(df_diputados_norm["periodo_id"], "periodo_id")

df_diputados_norm["fecha_nacimiento"] = normalizar_fecha(df_diputados_norm["fecha_nacimiento"], "fecha_nacimiento")

# fecha_inicio_periodo y fecha_termino_periodo se normalizan a datetime,
# pero su interpretación (Regla #6 y la anomalía detectada en F2_02) se
# resuelve en la sección 5, no aquí.
df_diputados_norm["fecha_inicio_periodo"] = normalizar_fecha(df_diputados_norm["fecha_inicio_periodo"], "fecha_inicio_periodo")
df_diputados_norm["fecha_termino_periodo"] = normalizar_fecha(df_diputados_norm["fecha_termino_periodo"], "fecha_termino_periodo")

df_diputados_norm.dtypes

diputado_id                         str
nombre                              str
nombre2                         float64
apellido_paterno                    str
apellido_materno                    str
fecha_nacimiento         datetime64[us]
rut                             float64
rut_dv                          float64
sexo_valor                        int64
sexo_desc                           str
periodo_id                          str
fecha_inicio_periodo     datetime64[us]
fecha_termino_periodo     datetime64[s]
dtype: object

### 4.2 · Militancias

In [7]:
df_militancias_norm = df_militancias.copy()

df_militancias_norm["diputado_id"] = normalizar_identificador(df_militancias_norm["diputado_id"], "diputado_id")
df_militancias_norm["partido_id"] = normalizar_identificador(df_militancias_norm["partido_id"], "partido_id")

df_militancias_norm["fecha_inicio"] = normalizar_fecha(df_militancias_norm["fecha_inicio"], "fecha_inicio")
df_militancias_norm["fecha_termino"] = normalizar_fecha(df_militancias_norm["fecha_termino"], "fecha_termino")

df_militancias_norm.dtypes

diputado_id                  str
partido_id                   str
partido_nombre               str
partido_alias                str
fecha_inicio      datetime64[us]
fecha_termino     datetime64[us]
dtype: object

### 4.3 · Detalle de votaciones

In [8]:
df_detalle_norm = df_detalle.copy()

df_detalle_norm["diputado_id"] = normalizar_identificador(df_detalle_norm["diputado_id"], "diputado_id")
df_detalle_norm["votacion_id"] = normalizar_identificador(df_detalle_norm["votacion_id"], "votacion_id")

df_detalle_norm["fecha"] = normalizar_fecha(df_detalle_norm["fecha"], "fecha", permitir_nulos=False)

for columna in ["total_si", "total_no", "total_abstencion", "total_dispensado"]:
    df_detalle_norm[columna] = normalizar_total(df_detalle_norm[columna], columna)

df_detalle_norm.dtypes

diputado_id                    str
nombre                         str
nombre2                    float64
apellido_paterno               str
apellido_materno               str
opcion_codigo                int64
opcion_voto                    str
votacion_id                    str
descripcion                    str
fecha               datetime64[us]
total_si                     Int64
total_no                     Int64
total_abstencion             Int64
total_dispensado             Int64
quorum_codigo                int64
quorum                         str
resultado_codigo             int64
resultado                      str
tipo_codigo                  int64
tipo                           str
dtype: object

### 4.4 · Catálogo del proyecto de ley

In [9]:
df_proyecto_norm = df_proyecto.copy()

df_proyecto_norm["numero_boletin"] = normalizar_identificador(df_proyecto_norm["numero_boletin"], "numero_boletin")
df_proyecto_norm["Id"] = normalizar_identificador(df_proyecto_norm["Id"], "Id")

df_proyecto_norm.dtypes

numero_boletin               str
Id                           str
Descripcion                  str
Fecha                        str
TotalSi                    int64
TotalNo                    int64
TotalAbstencion            int64
TotalDispensado            int64
Quorum                       str
Resultado                    str
Tipo                         str
TipoVotacionProyectoLey      str
Articulo                     str
TramiteConstitucional        str
TramiteReglamentario         str
dtype: object

In [10]:
df_proyecto_norm["Fecha"] = normalizar_fecha(df_proyecto_norm["Fecha"], "Fecha", permitir_nulos=False)

for columna in ["TotalSi", "TotalNo", "TotalAbstencion", "TotalDispensado"]:
    df_proyecto_norm[columna] = normalizar_total(df_proyecto_norm[columna], columna)

df_proyecto_norm.dtypes

numero_boletin                        str
Id                                    str
Descripcion                           str
Fecha                      datetime64[us]
TotalSi                             Int64
TotalNo                             Int64
TotalAbstencion                     Int64
TotalDispensado                     Int64
Quorum                                str
Resultado                             str
Tipo                                  str
TipoVotacionProyectoLey               str
Articulo                              str
TramiteConstitucional                 str
TramiteReglamentario                  str
dtype: object

### 4.5 · Periodos legislativos

In [11]:
df_periodos_norm = df_periodos.copy()

df_periodos_norm["periodo_id"] = normalizar_identificador(df_periodos_norm["periodo_id"], "periodo_id")
df_periodos_norm["fecha_inicio"] = normalizar_fecha(df_periodos_norm["fecha_inicio"], "fecha_inicio")
df_periodos_norm["fecha_termino"] = normalizar_fecha(df_periodos_norm["fecha_termino"], "fecha_termino")

df_periodos_norm.dtypes

periodo_id                  str
nombre                      str
fecha_inicio     datetime64[us]
fecha_termino    datetime64[us]
dtype: object

## 5. Procesamiento de diputados

Aplica las reglas 5 a 7 establecidas por el equipo

| # | Decisión | Aplicación |
|---|---|---|
| 5 | Unicidad | Exigir una sola fila por diputado_id |
| 6 | Periodo vigente | fecha_termino_periodo nula = periodo aún vigente; no sustituir |
| 7 | Diputados sin votos | Conservarlos y reportarlos; no crear filas artificiales |

**Nota sobre `fecha_inicio_periodo`/`fecha_termino_periodo`:** F2_02 detectó que 155 de 157
diputados tienen `fecha_inicio_periodo` igual al término del período 10, y
`fecha_termino_periodo` nula — un patrón que no coincide con la semántica esperada del
campo. El equipo acordó **no corregir ni reinterpretar este dato**: se documenta como
observación de calidad y no se utiliza como fuente de vigencia. Para eso se usa
`periodos.csv`, que sí es confiable.

### 5.1 Unicidad

In [12]:
duplicados_diputado = df_diputados_norm.duplicated(subset=["diputado_id"], keep=False)

print(f"Diputados duplicados (por diputado_id): {duplicados_diputado.sum()}")

if duplicados_diputado.any():
    display(df_diputados_norm[duplicados_diputado].sort_values("diputado_id"))

Diputados duplicados (por diputado_id): 0


### 5.2 Documentar la anomalía temporal (sin corregirla):

In [13]:
anomalia_fecha_inicio = df_diputados_norm["fecha_inicio_periodo"] == df_diputados_norm["fecha_inicio_periodo"].mode()[0]

print(f"Diputados con fecha_inicio_periodo = {df_diputados_norm['fecha_inicio_periodo'].mode()[0]}: {anomalia_fecha_inicio.sum()}")

for _, fila in df_diputados_norm[anomalia_fecha_inicio].iterrows():
    registrar_observacion(
        tabla="diputados",
        diputado_id=fila["diputado_id"],
        tipo="anomalia_temporal",
        descripcion=(
            "fecha_inicio_periodo coincide con el término del período 10 y "
            "fecha_termino_periodo es nula; no coincide con la semántica esperada del campo."
        ),
        regla_aplicada="No corregir; no usar como fuente de vigencia (acordado en equipo).",
    )

print(f"Observaciones registradas: {len(reporte_calidad)}")

Diputados con fecha_inicio_periodo = 2026-03-10 23:59:59: 155
Observaciones registradas: 155


### 5.3 Diputados sin votos

In [14]:
ids_diputados = set(df_diputados_norm["diputado_id"])
ids_con_voto = set(df_detalle_norm["diputado_id"])

sin_votos = ids_diputados - ids_con_voto

print(f"Diputados del catálogo sin voto registrado en las 15 votaciones: {len(sin_votos)}")

df_diputados_norm["tiene_votos"] = df_diputados_norm["diputado_id"].isin(ids_con_voto)

for did in sin_votos:
    fila = df_diputados_norm[df_diputados_norm["diputado_id"] == did].iloc[0]
    registrar_observacion(
        tabla="diputados",
        diputado_id=did,
        tipo="sin_votos",
        descripcion=f"{fila['nombre']} {fila['apellido_paterno']} no registra voto en ninguna de las 15 votaciones analizadas.",
        regla_aplicada="Conservar en la dimensión procesada; no crear filas artificiales (Regla #7).",
    )

display(df_diputados_norm[~df_diputados_norm["tiene_votos"]][["diputado_id", "nombre", "apellido_paterno", "apellido_materno"]])

Diputados del catálogo sin voto registrado en las 15 votaciones: 6


,diputado_id,nombre,apellido_paterno,apellido_materno
51,1031,Juan,Fuenzalida,Cobo
55,1130,Marta,González,Olea
64,1134,Andrés,Jouannet,Valderrama
124,1070,Patricio,Rosas,Barrientos
155,1184,Roberto,Celedón,Fernández
156,1185,Arturo,Barrios,Oteíza


### 5.4 Cerrar la tabla procesada:

**Columnas eliminadas de `diputados_procesados`:** `rut` y `rut_dv`  `nombre2` (0% de cobertura en la
fuente; la API nunca entrega este dato).

In [15]:
columnas_a_eliminar = ["rut", "rut_dv", "nombre2"]

diputados_procesados = df_diputados_norm.drop(columns=columnas_a_eliminar).copy()

print(f"diputados_procesados: {diputados_procesados.shape[0]} filas x {diputados_procesados.shape[1]} columnas")
diputados_procesados.head()

diputados_procesados: 157 filas x 11 columnas


,diputado_id,nombre,apellido_paterno,apellido_materno,fecha_nacimiento,sexo_valor,sexo_desc,periodo_id,fecha_inicio_periodo,fecha_termino_periodo,tiene_votos
0,1096,María Candelaria,Acevedo,Sáez,1958-09-12,0,Femenino,10,2026-03-10 23:59:59,NaT,True
1,1097,Eric,Aedo,Jeldres,1968-07-13,1,Masculino,10,2026-03-10 23:59:59,NaT,True
2,1098,Yovana,Ahumada,Palma,1973-02-24,0,Femenino,10,2026-03-10 23:59:59,NaT,True
3,1009,Jorge,Alessandri,Vergara,1979-06-08,1,Masculino,10,2026-03-10 23:59:59,NaT,True
4,803,René,Alinco,Bustos,1958-06-02,1,Masculino,10,2026-03-10 23:59:59,NaT,True


## 6. Procesamiento temporal de militancias

Construcción una línea de tiempo analítica sin ambigüedades por diputado, preservando en paralelo la evidencia observada.

### 6.1 · Intervalos inválidos

Un intervalo es inválido cuando `fecha_termino < fecha_inicio`. Estos registros se separan
como observación de calidad y se excluyen de la línea de tiempo utilizable, pero se
conservan en `militancias_observadas` para trazabilidad.

In [16]:
militancias_observadas = df_militancias_norm.copy()

intervalo_invalido = (
    militancias_observadas["fecha_termino"].notna()
    & (militancias_observadas["fecha_termino"] < militancias_observadas["fecha_inicio"])
)

militancias_observadas["intervalo_valido"] = ~intervalo_invalido
militancias_observadas["observacion_calidad"] = None

print(f"Intervalos inválidos detectados: {intervalo_invalido.sum()}")
display(militancias_observadas[intervalo_invalido])

Intervalos inválidos detectados: 1


,diputado_id,partido_id,partido_nombre,partido_alias,fecha_inicio,fecha_termino,intervalo_valido,observacion_calidad
380,1180,IND,Independientes,IND,2026-03-11,2026-03-10 23:59:59,False,None


### 6.2 · Detección de solapamientos

Se buscan pares de militancias del mismo diputado cuyos intervalos se superponen,
trabajando solo sobre los intervalos válidos (excluyendo el caso ya documentado en 6.1).
Se distingue si el solapamiento es del **mismo partido** o de
**partidos distintos** (requiere una regla de prioridad explícita).

In [17]:
militancias_validas = militancias_observadas[militancias_observadas["intervalo_valido"]].copy()

solapamientos = []

for diputado_id, grupo in militancias_validas.groupby("diputado_id"):
    grupo = grupo.sort_values("fecha_inicio").reset_index()
    for i in range(len(grupo) - 1):
        fila_a = grupo.iloc[i]
        fila_b = grupo.iloc[i + 1]

        termino_a = fila_a["fecha_termino"] if pd.notna(fila_a["fecha_termino"]) else pd.Timestamp.max
        inicio_b = fila_b["fecha_inicio"]

        if inicio_b <= termino_a:
            solapamientos.append({
                "diputado_id": diputado_id,
                "partido_a": fila_a["partido_nombre"],
                "partido_id_a": fila_a["partido_id"],
                "fecha_inicio_a": fila_a["fecha_inicio"],
                "fecha_termino_a": fila_a["fecha_termino"],
                "partido_b": fila_b["partido_nombre"],
                "partido_id_b": fila_b["partido_id"],
                "fecha_inicio_b": fila_b["fecha_inicio"],
                "fecha_termino_b": fila_b["fecha_termino"],
                "mismo_partido": fila_a["partido_id"] == fila_b["partido_id"],
            })

solapamientos_df = pd.DataFrame(solapamientos)
print(f"Solapamientos detectados: {len(solapamientos_df)}")
display(solapamientos_df)

Solapamientos detectados: 2


,diputado_id,partido_a,partido_id_a,fecha_inicio_a,fecha_termino_a,partido_b,partido_id_b,fecha_inicio_b,fecha_termino_b,mismo_partido
0,1017,Unión Demócrata Independiente,UDI,2018-03-11,2025-03-18 23:59:59,Independientes,IND,2020-09-29,2022-03-10 23:59:59,False
1,1114,Federación Regionalista Verde Social,FRVS,2022-03-11,2023-06-12 23:59:59,Independientes,IND,2022-06-13,2024-07-02 23:59:59,False


In [18]:
for did in ["1017", "1114"]:
    print(f"--- Diputado {did} ---")
    display(militancias_validas[militancias_validas["diputado_id"] == did][
        ["partido_id", "partido_nombre", "fecha_inicio", "fecha_termino"]
    ].sort_values("fecha_inicio"))

--- Diputado 1017 ---


,partido_id,partido_nombre,fecha_inicio,fecha_termino
70,UDI,Unión Demócrata Independiente,2018-03-11,2025-03-18 23:59:59
71,IND,Independientes,2020-09-29,2022-03-10 23:59:59
72,UDI,Unión Demócrata Independiente,2022-03-11,2025-03-18 23:59:59
73,IND,Independientes,2025-03-19,2026-03-10 23:59:59
74,PREP,Partido Republicano,2026-03-11,2030-03-10 23:59:59


--- Diputado 1114 ---


,partido_id,partido_nombre,fecha_inicio,fecha_termino
57,FRVS,Federación Regionalista Verde Social,2022-03-11,2023-06-12 23:59:59
58,IND,Independientes,2022-06-13,2024-07-02 23:59:59
59,FA,Frente Amplio,2024-07-03,2026-03-10 23:59:59
60,FA,Frente Amplio,2026-03-11,2030-03-10 23:59:59


### 6.3 · Aplicación de reglas particulares (Carter, Bugueño, Veloso)

Decisiones acordadas explícitamente con el equipo para los 3 casos detectados en 6.1 y 6.2.

**Carter (1017).** Durante el período en que se realizaron las votaciones del proyecto de
ley (2023), Carter presentan dos registos de militancia que cubren el período (filas 70 y 72). Como ambos registros indican que militaba por UDI, se usará el último para
efectos del análisis. Se conserva UDI (2022-03-11 → 2025-03-18)

**Bugueño (1114).** El criterio aplicado fue:

1. Es probable que la fecha de inicio de la militancia Independiente (fila 58) tenga un error en el origen, dado que si fuera 2023 coincidiría exactamente con el día posterior al último día de la militancia en FRVS (fila 57).
2. 14 de las 15 votaciones del proyecto de ley se realizaron durante el mes de mayo de 2023 y coinciden con la militancia en FRVS; la 15ª votación (2024-08-26) cae en el tramo que la fuente registra sucesivamente como Independiente y luego Frente Amplio.
3. Así mismo, para el análisis, es más útil representar su militancia en FRVS para determinar su posición política que las militancias sucesivas registradas en el mismo tramo.

Por el mismo criterio de coherencia aplicado a Veloso, se conserva FRVS, extendida hasta el final del período relevante (2022-03-11 → 2026-03-10), en vez de limitarla a su fecha de término observada.

**Veloso (1180).** Para 14 de las 15 votaciones analizadas, la fecha de votación coincide con su militancia observada en Revolución Democrática (RD); solo hay 1 voto durante el tramo
que la fuente registra como Independiente (verificado en el código siguiente). Por coherencia del análisis y para representar mejor su posición política, se decide mantener
su militancia en RD para las 15 votaciones, extendiendo la vigencia de RD hasta el final del período relevante.

In [19]:
def asignar_militancia_observada(diputado_id, tabla_militancias):
    """Cruza cada voto del diputado (df_detalle_norm) con la militancia vigente en esa
    fecha segun tabla_militancias, en vez de asumir un corte de fecha fijo."""
    votos = df_detalle_norm.loc[
        df_detalle_norm["diputado_id"] == diputado_id,
        ["votacion_id", "diputado_id", "fecha"],
    ].sort_values("fecha").reset_index(drop=True)

    intervalos = tabla_militancias.loc[
        tabla_militancias["diputado_id"] == diputado_id,
        ["diputado_id", "partido_id", "fecha_inicio", "fecha_termino"],
    ].sort_values("fecha_inicio")

    cruce = votos.merge(intervalos, on="diputado_id", how="left")

    cruce["vigente"] = (cruce["fecha_inicio"] <= cruce["fecha"]) & (
        cruce["fecha_termino"].isna() | (cruce["fecha"] <= cruce["fecha_termino"])
    )
    cruce["vigente"] = cruce["vigente"].fillna(False)

    def resumir(grupo):
        vigentes = grupo.loc[grupo["vigente"], "partido_id"]
        if len(vigentes) == 0:
            return "SIN_MILITANCIA_OBSERVADA"
        if len(vigentes) > 1:
            return "AMBIGUA(" + "/".join(vigentes) + ")"
        return vigentes.iloc[0]

    resumen_por_votacion = cruce.groupby("votacion_id").apply(resumir, include_groups=False)
    votos["militancia_observada"] = votos["votacion_id"].map(resumen_por_votacion)
    return votos


In [20]:
votos_bugueno = asignar_militancia_observada("1114", militancias_validas)

print(f"Votaciones de Bugueno: {len(votos_bugueno)}")
print(votos_bugueno["militancia_observada"].value_counts())
display(votos_bugueno)

Votaciones de Bugueno: 15
militancia_observada
AMBIGUA(FRVS/IND)    14
FA                    1
Name: count, dtype: int64


,votacion_id,diputado_id,fecha,militancia_observada
0,20627,1114,2023-05-08 19:05:22,AMBIGUA(FRVS/IND)
1,20628,1114,2023-05-08 19:06:49,AMBIGUA(FRVS/IND)
2,20629,1114,2023-05-08 19:08:21,AMBIGUA(FRVS/IND)
3,20630,1114,2023-05-08 19:09:23,AMBIGUA(FRVS/IND)
4,20631,1114,2023-05-08 19:10:13,AMBIGUA(FRVS/IND)
5,20632,1114,2023-05-08 19:11:06,AMBIGUA(FRVS/IND)
6,20633,1114,2023-05-08 19:11:56,AMBIGUA(FRVS/IND)
7,20634,1114,2023-05-08 19:12:58,AMBIGUA(FRVS/IND)
8,20635,1114,2023-05-08 19:13:51,AMBIGUA(FRVS/IND)
9,20636,1114,2023-05-08 19:15:04,AMBIGUA(FRVS/IND)


In [21]:
votos_veloso = asignar_militancia_observada("1180", militancias_validas)

print(f"Votaciones de Veloso: {len(votos_veloso)}")
print(votos_veloso["militancia_observada"].value_counts())
display(votos_veloso)

Votaciones de Veloso: 15
militancia_observada
RD     14
IND     1
Name: count, dtype: int64


,votacion_id,diputado_id,fecha,militancia_observada
0,20627,1180,2023-05-08 19:05:22,RD
1,20628,1180,2023-05-08 19:06:49,RD
2,20629,1180,2023-05-08 19:08:21,RD
3,20630,1180,2023-05-08 19:09:23,RD
4,20631,1180,2023-05-08 19:10:13,RD
5,20632,1180,2023-05-08 19:11:06,RD
6,20633,1180,2023-05-08 19:11:56,RD
7,20634,1180,2023-05-08 19:12:58,RD
8,20635,1180,2023-05-08 19:13:51,RD
9,20636,1180,2023-05-08 19:15:04,RD


In [22]:
militancias_analiticas = militancias_validas.copy()
militancias_analiticas["fecha_termino_original"] = militancias_analiticas["fecha_termino"]
militancias_analiticas["regla_asignacion_partido"] = "vigencia_temporal_estandar"

# Fin del período 10 (2022-2026); se usa para extender FRVS (Bugueño) y RD (Veloso).
fecha_fin_periodo_10 = df_periodos_norm.loc[df_periodos_norm["periodo_id"] == "10", "fecha_termino"].iloc[0]

# --- Carter (1017): conservar solo filas 72 (UDI) y 74 (PREP) ---
filas_excluir_carter = [70, 71, 73]
for idx in filas_excluir_carter:
    fila = militancias_observadas.loc[idx]
    registrar_observacion(
        tabla="militancias",
        diputado_id=fila["diputado_id"],
        tipo="excepcion_particular",
        descripcion=f"Registro {fila['partido_nombre']} ({fila['fecha_inicio']}→{fila['fecha_termino']}) excluido de la línea de tiempo analítica.",
        regla_aplicada="Caso particular Carter: conservar únicamente UDI 2022-2025 y PREP 2026-2030.",
    )

# --- Bugueño (1114): extender FRVS (fila 57) hasta el fin del período; excluir 58, 59 y 60 ---
militancias_analiticas.loc[57, "fecha_termino"] = fecha_fin_periodo_10
militancias_analiticas.loc[57, "regla_asignacion_partido"] = "extension_caso_particular_bugueno"

filas_excluir_bugueno = [58, 59, 60]
for idx in filas_excluir_bugueno:
    fila = militancias_observadas.loc[idx]
    registrar_observacion(
        tabla="militancias",
        diputado_id=fila["diputado_id"],
        tipo="excepcion_particular",
        descripcion=f"Registro {fila['partido_nombre']} ({fila['fecha_inicio']}→{fila['fecha_termino']}) absorbido por la extensión de FRVS hasta el fin del período (14 de 15 votaciones coinciden con FRVS; la 15ª, del 2024-08-26, queda cubierta por la extensión).",
        regla_aplicada="Caso particular Bugueño: extender FRVS hasta el fin del período relevante, mismo criterio que Veloso; excluir Independiente y Frente Amplio (x2).",
    )

indices_excluir = filas_excluir_carter + filas_excluir_bugueno

# --- Veloso (1180): extender RD (377) hasta 2026-03-10; excluir IND (378) ---
militancias_analiticas.loc[377, "fecha_termino"] = fecha_fin_periodo_10
militancias_analiticas.loc[377, "regla_asignacion_partido"] = "extension_caso_particular_veloso"

fila_ind_veloso = militancias_observadas.loc[378]
registrar_observacion(
    tabla="militancias",
    diputado_id=fila_ind_veloso["diputado_id"],
    tipo="excepcion_particular",
    descripcion=f"Registro IND ({fila_ind_veloso['fecha_inicio']}→{fila_ind_veloso['fecha_termino']}) absorbido por la extensión de RD (14/15 votaciones coinciden con RD).",
    regla_aplicada="Caso particular Veloso: extender RD hasta el fin del período relevante; excluir IND.",
)

indices_excluir.append(378)

# Aplicar exclusiones
militancias_analiticas = militancias_analiticas.drop(index=[i for i in indices_excluir if i in militancias_analiticas.index])

print(f"militancias_analiticas: {len(militancias_analiticas)} filas (se excluyeron {len(indices_excluir)} por reglas particulares)")
display(militancias_analiticas[militancias_analiticas["diputado_id"].isin(["1017", "1114", "1180"])])

militancias_analiticas: 399 filas (se excluyeron 7 por reglas particulares)


,diputado_id,partido_id,partido_nombre,partido_alias,fecha_inicio,fecha_termino,intervalo_valido,observacion_calidad,fecha_termino_original,regla_asignacion_partido
57,1114,FRVS,Federación Regionalista Verde Social,FRVS,2022-03-11,2026-03-10 23:59:59,True,None,2023-06-12 23:59:59,extension_caso_particular_bugueno
72,1017,UDI,Unión Demócrata Independiente,UDI,2022-03-11,2025-03-18 23:59:59,True,None,2025-03-18 23:59:59,vigencia_temporal_estandar
74,1017,PREP,Partido Republicano,PREP,2026-03-11,2030-03-10 23:59:59,True,None,2030-03-10 23:59:59,vigencia_temporal_estandar
377,1180,RD,Revolución Democrática,RD,2022-03-11,2026-03-10 23:59:59,True,None,2024-05-30 23:59:59,extension_caso_particular_veloso
379,1180,FA,Frente Amplio,FA,2026-03-11,2030-03-10 23:59:59,True,None,2030-03-10 23:59:59,vigencia_temporal_estandar


In [23]:
solapamientos_finales = []

for diputado_id, grupo in militancias_analiticas.groupby("diputado_id"):
    grupo = grupo.sort_values("fecha_inicio").reset_index()
    for i in range(len(grupo) - 1):
        fila_a = grupo.iloc[i]
        fila_b = grupo.iloc[i + 1]

        termino_a = fila_a["fecha_termino"] if pd.notna(fila_a["fecha_termino"]) else pd.Timestamp.max
        inicio_b = fila_b["fecha_inicio"]

        if inicio_b <= termino_a:
            solapamientos_finales.append({
                "diputado_id": diputado_id,
                "partido_a": fila_a["partido_nombre"],
                "fecha_termino_a": fila_a["fecha_termino"],
                "partido_b": fila_b["partido_nombre"],
                "fecha_inicio_b": fila_b["fecha_inicio"],
            })

print(f"Solapamientos remanentes en militancias_analiticas: {len(solapamientos_finales)}")
if solapamientos_finales:
    display(pd.DataFrame(solapamientos_finales))

Solapamientos remanentes en militancias_analiticas: 0


## 7. Procesamiento del detalle de votaciones

Aplica las reglas 15 a 17 de decisiones:

| # | Decisión | Aplicación |
|---|---|---|
| 15 | Granularidad | Exigir una fila por combinación votacion_id + diputado_id |
| 16 | Conservación | Mantener las columnas del detalle, salvo `nombre2` (0% de cobertura, igual que en `diputados_procesados`); no seleccionar otro subconjunto |
| 17 | Identidad | Verificar que los nombres coincidan con la tabla maestra; reportar sin sobrescribir |

### 7.1 · Granularidad

In [24]:
duplicados_detalle = df_detalle_norm.duplicated(subset=["votacion_id", "diputado_id"], keep=False)

print(f"Filas duplicadas (por votacion_id + diputado_id): {duplicados_detalle.sum()}")

if duplicados_detalle.any():
    display(df_detalle_norm[duplicados_detalle].sort_values(["votacion_id", "diputado_id"]))

Filas duplicadas (por votacion_id + diputado_id): 0


### 7.2 · Verificación de identidad (nombres vs. tabla maestra)

In [25]:
detalle_con_maestra = df_detalle_norm.merge(
    df_diputados_norm[["diputado_id", "nombre", "apellido_paterno", "apellido_materno"]],
    on="diputado_id",
    suffixes=("_detalle", "_maestra"),
    how="left",
)

diferencia_nombre = detalle_con_maestra["nombre_detalle"] != detalle_con_maestra["nombre_maestra"]
diferencia_paterno = detalle_con_maestra["apellido_paterno_detalle"] != detalle_con_maestra["apellido_paterno_maestra"]
diferencia_materno = detalle_con_maestra["apellido_materno_detalle"] != detalle_con_maestra["apellido_materno_maestra"]

cualquier_diferencia = diferencia_nombre | diferencia_paterno | diferencia_materno

print(f"Filas con diferencia de nombre respecto a la tabla maestra: {cualquier_diferencia.sum()}")

if cualquier_diferencia.any():
    columnas_comparar = [
        "diputado_id", "votacion_id",
        "nombre_detalle", "nombre_maestra",
        "apellido_paterno_detalle", "apellido_paterno_maestra",
        "apellido_materno_detalle", "apellido_materno_maestra",
    ]
    diferencias_df = detalle_con_maestra[cualquier_diferencia][columnas_comparar].drop_duplicates(subset=["diputado_id"])
    display(diferencias_df)

    for _, fila in diferencias_df.iterrows():
        registrar_observacion(
            tabla="detalle_votaciones",
            diputado_id=fila["diputado_id"],
            tipo="diferencia_identidad",
            descripcion=(
                f"Detalle: '{fila['nombre_detalle']} {fila['apellido_paterno_detalle']} {fila['apellido_materno_detalle']}' "
                f"vs. Maestra: '{fila['nombre_maestra']} {fila['apellido_paterno_maestra']} {fila['apellido_materno_maestra']}'"
            ),
            regla_aplicada="Reportar sin sobrescribir; se conserva el dato de cada fuente (Regla #17).",
        )

Filas con diferencia de nombre respecto a la tabla maestra: 0


### 7.3 · Cerrar la tabla procesada

In [26]:
# Regla #16: se conservan las columnas originales del detalle, sin seleccionar subconjunto,
# salvo `nombre2`: tiene 0% de cobertura en la fuente (igual que en diputados_procesados,
# ver 5.4) y se elimina en ambas tablas por consistencia.
detalle_votaciones_procesado = df_detalle_norm.drop(columns=["nombre2"]).copy()

print(f"detalle_votaciones_procesado: {detalle_votaciones_procesado.shape[0]} filas x {detalle_votaciones_procesado.shape[1]} columnas")
detalle_votaciones_procesado.columns.tolist()

detalle_votaciones_procesado: 1993 filas x 19 columnas


['diputado_id',
 'nombre',
 'apellido_paterno',
 'apellido_materno',
 'opcion_codigo',
 'opcion_voto',
 'votacion_id',
 'descripcion',
 'fecha',
 'total_si',
 'total_no',
 'total_abstencion',
 'total_dispensado',
 'quorum_codigo',
 'quorum',
 'resultado_codigo',
 'resultado',
 'tipo_codigo',
 'tipo']

## 8. Procesamiento del catálogo del proyecto de ley

Aplica las reglas 18 a 21 de decisiones del equipo:

| # | Decisión | Aplicación |
|---|---|---|
| 18 | Unicidad | Exigir una sola fila por votacion_id en proyecto_ley |
| 19 | Cobertura | Toda votación del detalle debe existir en proyecto_ley |
| 20 | Consistencia | Validar igualdad de fecha, totales, quórum, resultado y tipo |
| 21 | Atributos nuevos | Preparar campos adicionales para su incorporación |

### 8.1 · Unicidad

In [27]:
duplicados_proyecto = df_proyecto_norm.duplicated(subset=["Id"], keep=False)

print(f"Votaciones duplicadas (por Id): {duplicados_proyecto.sum()}")

if duplicados_proyecto.any():
    display(df_proyecto_norm[duplicados_proyecto])

Votaciones duplicadas (por Id): 0


### 8.2 · Cobertura

In [28]:
ids_proyecto = set(df_proyecto_norm["Id"])
ids_detalle = set(detalle_votaciones_procesado["votacion_id"])

detalle_sin_catalogo = ids_detalle - ids_proyecto
catalogo_sin_detalle = ids_proyecto - ids_detalle

print(f"Votaciones en detalle sin catálogo: {len(detalle_sin_catalogo)}")
print(f"Votaciones en catálogo sin detalle: {len(catalogo_sin_detalle)}")

for votacion_id in catalogo_sin_detalle:
    registrar_observacion(
        tabla="proyecto_ley",
        diputado_id=None,
        tipo="sin_detalle",
        descripcion=f"Votación {votacion_id} está en el catálogo pero no tiene detalle nominal.",
        regla_aplicada="Reportar; no crear filas artificiales (Regla #19).",
    )

Votaciones en detalle sin catálogo: 0
Votaciones en catálogo sin detalle: 0


### 8.3: consistencia

In [29]:
mapeo_columnas = {
    "Fecha": "fecha",
    "TotalSi": "total_si",
    "TotalNo": "total_no",
    "TotalAbstencion": "total_abstencion",
    "TotalDispensado": "total_dispensado",
    "Quorum": "quorum",
    "Resultado": "resultado",
    "Tipo": "tipo",
}

# Un valor por votación desde el detalle (todas las filas de una misma votación
# deben compartir estos metadatos, ya validado implícitamente en F2_02).
metadata_detalle = detalle_votaciones_procesado.groupby("votacion_id")[
    list(mapeo_columnas.values())
].first().reset_index()

comparacion = df_proyecto_norm.merge(
    metadata_detalle,
    left_on="Id",
    right_on="votacion_id",
    suffixes=("_proyecto", "_detalle"),
)

inconsistencias = []

for columna_proyecto, columna_detalle in mapeo_columnas.items():
    col_p = f"{columna_proyecto}_proyecto" if columna_proyecto in comparacion.columns and columna_detalle in comparacion.columns and columna_proyecto == columna_detalle else columna_proyecto
    col_d = columna_detalle if columna_detalle not in df_proyecto_norm.columns else f"{columna_detalle}_detalle"

    # Ajuste: como los nombres no colisionan (PascalCase vs snake_case), no hay sufijo real.
    diferentes = comparacion[columna_proyecto] != comparacion[columna_detalle]
    if diferentes.any():
        for _, fila in comparacion[diferentes].iterrows():
            inconsistencias.append({
                "votacion_id": fila["Id"],
                "campo": columna_proyecto,
                "valor_proyecto": fila[columna_proyecto],
                "valor_detalle": fila[columna_detalle],
            })

print(f"Inconsistencias detectadas: {len(inconsistencias)}")
if inconsistencias:
    display(pd.DataFrame(inconsistencias))

Inconsistencias detectadas: 0


### 8.4 · Atributos nuevos para incorporación

In [30]:
columnas_nuevas = ["numero_boletin", "TipoVotacionProyectoLey", "Articulo", "TramiteConstitucional", "TramiteReglamentario"]

print("Disponibilidad de atributos nuevos en proyecto_ley_procesado:")
for col in columnas_nuevas:
    nulos = df_proyecto_norm[col].isna().sum()
    print(f"  {col}: {df_proyecto_norm[col].notna().sum()} no nulos, {nulos} nulos")

Disponibilidad de atributos nuevos en proyecto_ley_procesado:
  numero_boletin: 15 no nulos, 0 nulos
  TipoVotacionProyectoLey: 15 no nulos, 0 nulos
  Articulo: 14 no nulos, 1 nulos
  TramiteConstitucional: 15 no nulos, 0 nulos
  TramiteReglamentario: 15 no nulos, 0 nulos


In [31]:
proyecto_ley_procesado = df_proyecto_norm.copy()

print(f"proyecto_ley_procesado: {proyecto_ley_procesado.shape[0]} filas x {proyecto_ley_procesado.shape[1]} columnas")
proyecto_ley_procesado.head()

proyecto_ley_procesado: 15 filas x 15 columnas


,numero_boletin,Id,Descripcion,Fecha,TotalSi,TotalNo,TotalAbstencion,TotalDispensado,Quorum,Resultado,Tipo,TipoVotacionProyectoLey,Articulo,TramiteConstitucional,TramiteReglamentario
0,11092-07,42724,Boletín N°11092-07,2024-08-26 19:03:25,65,22,36,0,Quórum Calificado,Aprobado,Proyecto de Ley,Única,Proposición de la Comisión Mixta recaída en el...,Comisión Mixta,Sin Informe
1,11092-07,21219,Boletín N°11092-07,2023-05-08 19:18:45,74,59,2,0,Quórum Simple,Aprobado,Proyecto de Ley,Particular,"Letra c) del artículo 35, contenido en el nume...",Segundo Trámite,Primer Informe
2,11092-07,21218,Boletín N°11092-07,2023-05-08 19:17:43,73,60,2,0,Quórum Simple,Aprobado,Proyecto de Ley,Particular,"Letra b) del artículo 35, contenido en el nume...",Segundo Trámite,Primer Informe
3,11092-07,21217,Boletín N°11092-07,2023-05-08 19:16:53,90,38,7,0,Quórum Simple,Aprobado,Proyecto de Ley,Particular,"Artículo 32, contenido en el numeral 12) del a...",Segundo Trámite,Primer Informe
4,11092-07,20637,Boletín N°11092-07,2023-05-08 19:15:59,91,41,3,0,Quórum Simple,Aprobado,Proyecto de Ley,Particular,"Artículo 31, contenido en el numeral 12) del a...",Segundo Trámite,Primer Informe


## 9. Entregables

Se guardan las 6 tablas procesadas en `data/processed/`, listas para la fase de
integración. Los archivos originales en `data/interim/` no se modifican.

In [32]:
entregables = {
    "diputados_procesados.csv": diputados_procesados,
    "militancias_observadas.csv": militancias_observadas,
    "militancias_analiticas.csv": militancias_analiticas,
    "detalle_votaciones_procesado.csv": detalle_votaciones_procesado,
    "proyecto_ley_procesado.csv": proyecto_ley_procesado,
    "reporte_calidad.csv": pd.DataFrame(reporte_calidad),
}

for nombre_archivo, df in entregables.items():
    ruta = PROCESSED_DIR / nombre_archivo
    df.to_csv(ruta, index=False, encoding="utf-8")
    print(f"Guardado: {ruta} ({len(df)} filas)")

Guardado: C:\Users\joign\OneDrive\Desktop\Magister en Ciencia de Datos e IA\1. Programación para la ciencia de datos\grupo_1_programacion_ciencia_de_datos\F2\data\processed\diputados_procesados.csv (157 filas)
Guardado: C:\Users\joign\OneDrive\Desktop\Magister en Ciencia de Datos e IA\1. Programación para la ciencia de datos\grupo_1_programacion_ciencia_de_datos\F2\data\processed\militancias_observadas.csv (407 filas)
Guardado: C:\Users\joign\OneDrive\Desktop\Magister en Ciencia de Datos e IA\1. Programación para la ciencia de datos\grupo_1_programacion_ciencia_de_datos\F2\data\processed\militancias_analiticas.csv (399 filas)
Guardado: C:\Users\joign\OneDrive\Desktop\Magister en Ciencia de Datos e IA\1. Programación para la ciencia de datos\grupo_1_programacion_ciencia_de_datos\F2\data\processed\detalle_votaciones_procesado.csv (1993 filas)
Guardado: C:\Users\joign\OneDrive\Desktop\Magister en Ciencia de Datos e IA\1. Programación para la ciencia de datos\grupo_1_programacion_ciencia_d

In [33]:
reporte_calidad_df = pd.DataFrame(reporte_calidad)
print(f"Total de observaciones registradas: {len(reporte_calidad_df)}")
reporte_calidad_df["tipo"].value_counts()

Total de observaciones registradas: 168


tipo
anomalia_temporal       155
excepcion_particular      7
sin_votos                 6
Name: count, dtype: int64